In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [4]:
import numpy as np
import pandas as pd
import os
import warnings
import joblib
import gc

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

import xgboost as xgb
import mlflow
import dagshub

warnings.filterwarnings("ignore")
print("All imports OK ✓")

All imports OK ✓


**load and merge data. clean, reduce memory**

In [5]:
path = "/kaggle/input/competitions/ieee-fraud-detection/"

print("Loading data...")
train_tr = pd.read_csv(path + "train_transaction.csv")
train_id = pd.read_csv(path + "train_identity.csv")
test_tr  = pd.read_csv(path + "test_transaction.csv")
test_id  = pd.read_csv(path + "test_identity.csv")
print(f"  train_transaction: {train_tr.shape}")
print(f"  train_identity:    {train_id.shape}")
print(f"  test_transaction:  {test_tr.shape}")
print(f"  test_identity:     {test_id.shape}")

print("\nFixing column names (hyphen → underscore)...")
train_id.columns = [c.replace("-", "_") for c in train_id.columns]
test_id.columns  = [c.replace("-", "_") for c in test_id.columns]

print("\nMerging...")
train = train_tr.merge(train_id, on="TransactionID", how="left")
test  = test_tr.merge(test_id,   on="TransactionID", how="left")
print(f"  train merged: {train.shape}")
print(f"  test merged:  {test.shape}")

X = train.drop(columns=["isFraud", "TransactionID"])
y = train["isFraud"]
test = test.drop(columns=["TransactionID"])

print("\nAligning columns...")
only_in_train = set(X.columns) - set(test.columns)
only_in_test  = set(test.columns) - set(X.columns)
print(f"  Cols only in train: {len(only_in_train)}")
print(f"  Cols only in test:  {len(only_in_test)}")
common_cols = [c for c in X.columns if c in test.columns]
X    = X[common_cols]
test = test[common_cols]
assert list(X.columns) == list(test.columns)
print(f"  Aligned: X={X.shape}, test={test.shape}")

print("\nDropping high missing (>50%)...")
missing_rate  = X.isnull().mean()
high_missing  = missing_rate[missing_rate > 0.5].index.tolist()
X    = X.drop(columns=high_missing)
test = test.drop(columns=high_missing)
print(f"  Dropped {len(high_missing)} cols → X={X.shape}")

print("\nDropping near-zero variance (std < 0.01)...")
num_cols_temp = X.select_dtypes(include=[np.number]).columns
low_var = [c for c in num_cols_temp if X[c].std() < 0.01]
X    = X.drop(columns=low_var)
test = test.drop(columns=low_var)
print(f"  Dropped {len(low_var)} cols → X={X.shape}")

print("\nReducing memory...")
def reduce_mem(df, name):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    print(f"  {name}: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
    return df

X    = reduce_mem(X,    "X")
test = reduce_mem(test, "test")
print(f"  Fraud rate: {y.mean():.4f} ({y.sum()} fraud out of {len(y)} total)")

Loading data...
  train_transaction: (590540, 394)
  train_identity:    (144233, 41)
  test_transaction:  (506691, 393)
  test_identity:     (141907, 41)

Fixing column names (hyphen → underscore)...

Merging...
  train merged: (590540, 434)
  test merged:  (506691, 433)

Aligning columns...
  Cols only in train: 0
  Cols only in test:  0
  Aligned: X=(590540, 432), test=(506691, 432)

Dropping high missing (>50%)...
  Dropped 214 cols → X=(590540, 218)

Dropping near-zero variance (std < 0.01)...
  Dropped 2 cols → X=(590540, 216)

Reducing memory...
  X: 704.8 MB
  test: 607.7 MB
  Fraud rate: 0.0350 (20663 fraud out of 590540 total)


**MLflow / DagsHub setup**

In [6]:
print("Setting up MLflow + DagsHub...")
os.environ["MLFLOW_TRACKING_USERNAME"] = "kende23"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "be7a74e194c6b7c0d666a4938cfee49142c7d603"
mlflow.set_tracking_uri("https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow")
dagshub.init(repo_owner="kende23", repo_name="ieee-cis-fraud-detection", mlflow=True)

experiment_name = "xgboost_experiments"
if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(experiment_name)
    print(f"Created experiment: {experiment_name}")
else:
    print(f"Found existing experiment: {experiment_name}")

mlflow.set_experiment(experiment_name)
print("MLflow + DagsHub connected ✓")

Setting up MLflow + DagsHub...


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=a3d7dfe7-c577-40f9-8b2e-5c10c35d1ab6&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=4f7c2606f6a2773f55735d601b014cb890f592488a9a8cb0546e312e4b19375e




Accessing as kende23

Initialized MLflow to track repo "kende23/ieee-cis-fraud-detection"

Repository kende23/ieee-cis-fraud-detection initialized!

Found existing experiment: xgboost_experiments
MLflow + DagsHub connected ✓


**Split**

In [7]:
print("=== TRAIN/VAL SPLIT ===")
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"  Train fraud rate: {y_train.mean():.4f}")
print(f"  Val   fraud rate: {y_val.mean():.4f}")

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
print(f"\n  Numeric cols:     {len(num_cols)}")
print(f"  Categorical cols: {len(cat_cols)}")

del X
gc.collect()
print("\nFreed X from memory ✓")

=== TRAIN/VAL SPLIT ===
  X_train: (472432, 216), y_train: (472432,)
  X_val:   (118108, 216),   y_val:   (118108,)
  Train fraud rate: 0.0350
  Val   fraud rate: 0.0350

  Numeric cols:     207
  Categorical cols: 9

Freed X from memory ✓


**Build XGBoost pipeline & train**

In [8]:
print("=== BUILDING XGBOOST PIPELINE ===")

# XGBoost handles missing values natively so no imputer needed for numerics
# Use OrdinalEncoder instead of OHE — much cheaper memory-wise
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

# scale_pos_weight handles class imbalance for XGBoost
fraud_count     = int(y_train.sum())
nonfraud_count  = int((y_train == 0).sum())
scale_pos_weight = nonfraud_count / fraud_count
print(f"  scale_pos_weight: {scale_pos_weight:.2f} (fraud={fraud_count}, non-fraud={nonfraud_count})")

model_xgb = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="auc",
        tree_method="hist",       # faster training
        device="cuda" if os.path.exists("/usr/local/cuda") else "cpu",
        n_jobs=-1
    ))
])

print("Pipeline built ✓")
print("Training XGBoost... (this may take 5-10 minutes)")
model_xgb.fit(X_train, y_train)
print("Training done ✓")

del X_train
gc.collect()
print("Freed X_train from memory ✓")

=== BUILDING XGBOOST PIPELINE ===
  scale_pos_weight: 27.58 (fraud=16530, non-fraud=455902)
Pipeline built ✓
Training XGBoost... (this may take 5-10 minutes)
Training done ✓
Freed X_train from memory ✓


**Evaluate & log**

In [9]:
print("=== EVALUATING XGBOOST ===")

val_probs = model_xgb.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)

val_auc       = roc_auc_score(y_val, val_probs)
val_precision = precision_score(y_val, val_preds)
val_recall    = recall_score(y_val, val_preds)
val_f1        = f1_score(y_val, val_preds)

print(f"\n=== VALIDATION METRICS ===")
print(f"  AUC:       {val_auc:.4f}")
print(f"  Precision: {val_precision:.4f}")
print(f"  Recall:    {val_recall:.4f}")
print(f"  F1:        {val_f1:.4f}")

metrics = {
    "val_auc":       val_auc,
    "val_precision": val_precision,
    "val_recall":    val_recall,
    "val_f1":        val_f1,
}

params = {
    "n_estimators":      300,
    "max_depth":         6,
    "learning_rate":     0.05,
    "subsample":         0.8,
    "colsample_bytree":  0.8,
    "scale_pos_weight":  round(scale_pos_weight, 2),
    "missing_threshold": 0.5,
}

print("\nLogging to MLflow / DagsHub...")
with mlflow.start_run(run_name="xgboost_baseline"):
    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    joblib.dump(model_xgb, "xgb_model.pkl")
    mlflow.log_artifact("xgb_model.pkl")
    print(f"  Run ID: {mlflow.active_run().info.run_id}")

print("Logged to DagsHub ✓")
print("Check: https://dagshub.com/kende23/ieee-cis-fraud-detection")

del X_val
gc.collect()
print("Freed X_val from memory ✓")

=== EVALUATING XGBOOST ===

=== VALIDATION METRICS ===
  AUC:       0.9368
  Precision: 0.2425
  Recall:    0.8147
  F1:        0.3737

Logging to MLflow / DagsHub...
  Run ID: 0504eb8c66aa485a8429012b36a7eb36
🏃 View run xgboost_baseline at: https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow/#/experiments/2/runs/0504eb8c66aa485a8429012b36a7eb36
🧪 View experiment at: https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow/#/experiments/2
Logged to DagsHub ✓
Check: https://dagshub.com/kende23/ieee-cis-fraud-detection
Freed X_val from memory ✓


**Submission**

In [10]:
print("=== GENERATING SUBMISSION ===")
print(f"  test shape: {test.shape}")

print("Predicting...")
test_probs = model_xgb.predict_proba(test)[:, 1]

print(f"  Predictions shape: {test_probs.shape}")
print(f"  Predicted fraud rate: {test_probs.mean():.4f}")
print(f"  Min prob: {test_probs.min():.4f}")
print(f"  Max prob: {test_probs.max():.4f}")

submission = pd.DataFrame({
    "TransactionID": test_tr["TransactionID"].values,
    "isFraud": test_probs
})

print(f"\nSubmission shape: {submission.shape}")
print(submission.head(10))

submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv saved ✓")
print("Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions")

=== GENERATING SUBMISSION ===
  test shape: (506691, 216)
Predicting...
  Predictions shape: (506691,)
  Predicted fraud rate: 0.2242
  Min prob: 0.0010
  Max prob: 0.9994

Submission shape: (506691, 2)
   TransactionID   isFraud
0        3663549  0.089886
1        3663550  0.086905
2        3663551  0.129574
3        3663552  0.060989
4        3663553  0.049705
5        3663554  0.130911
6        3663555  0.257633
7        3663556  0.411482
8        3663557  0.017818
9        3663558  0.220695

submission.csv saved ✓
Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions
